## Balance dataset across each bins

In [1]:
import pandas as pd
import numpy as np

In [2]:
# --- CONFIGURATION ---
ORIGINAL_TRAIN = "/data/nas-gpu/wang/tmach007/SpectralSimilarityPredictor/data_splits/stratified_binary_07_dataset_train.feather"
MONA_SOURCE = "/data/nas-gpu/wang/tmach007/SpectralSimilarityPredictor/data_splits/mona_safe_augmentation_candidates.feather"
OUTPUT_PATH = "/data/nas-gpu/wang/tmach007/SpectralSimilarityPredictor/data_splits/stratified_binary_07_dataset_train_CUSTOM_BALANCED.feather"

In [3]:
# Metadata Paths (For Scaffold Lookup)
COMBINED_MOL = f"/data/nas-gpu/wang/tmach007/SpectralSimilarityPredictor/mass_spec_gym_data/mol_df_COMBINED.pkl"
COMBINED_SPEC = f"/data/nas-gpu/wang/tmach007/SpectralSimilarityPredictor/mass_spec_gym_data/spec_df_COMBINED.pkl"

In [5]:
# --- THE SHAPE DEFINITION ---
# Instead of fixed counts, we define RATIOS relative to the "Hard Peak".
# 1.0 = The maximum number of pairs found in the Hard zones.
SHAPE_RATIOS = {
    0: 0.25,  # 0.0-0.2 (Easy Neg): Keep to 25% of the peak height
    1: 0.50,  # 0.2-0.4: Ramp up
    2: 1.00,  # 0.4-0.6 (Hard): PEAK (Take everything)
    3: 1.00,  # 0.6-0.8 (Hard): PEAK (Take everything)
    4: 0.33   # 0.8-1.0 (Easy Pos): Keep to 33% of peak height
}

# The hard limits (we still cap the peak to avoid exploding memory)
GLOBAL_MAX_PEAK = 8000 
GLOBAL_MIN_PEAK = 500   # If a regime has fewer than 500 hard pairs, we force at least 500 (if data exists) to avoid starvation.

SIM_BINS = [
    (0.0, 0.2), (0.2, 0.4), (0.4, 0.6), (0.6, 0.8), (0.8, 1.0001)
]

In [6]:
# --- 1. LOAD DATA ---
print("Loading Data...")
df_train = pd.read_feather(ORIGINAL_TRAIN)
df_mona = pd.read_feather(MONA_SOURCE)

if 'label' not in df_mona.columns:
    df_mona['label'] = (df_mona['cosine_similarity'] >= 0.7).astype(int)

Loading Data...


In [7]:
# Load Metadata for Diversity
print("Loading Metadata...")
df_mol = pd.read_pickle(COMBINED_MOL)
df_spec = pd.read_pickle(COMBINED_SPEC)
mol2scaffold = dict(zip(df_mol['mol_id'], df_mol['scaffold']))
spec2mol = dict(zip(df_spec['spec_id'], df_spec['mol_id']))

Loading Metadata...


In [8]:
def get_scaffold_from_spec(spec_id):
    mol_id = spec2mol.get(spec_id)
    if mol_id is None: return "Unknown"
    scaff = mol2scaffold.get(mol_id)
    return scaff if scaff else "Generic"

# --- 2. PREPROCESSING ---
def get_regime(val):
    val = str(val)
    if "Isomers" in val: return "0. Exact Isomers"
    if "Isobaric" in val: return "1. Isobaric"
    if "Tiny" in val: return "2. Tiny"
    if "Medium" in val: return "3. Medium"
    if "Large" in val: return "4. Large"
    if "Huge" in val: return "5. Huge"
    return "Unknown"

In [9]:
def assign_sim_bin(sim):
    for i, (low, high) in enumerate(SIM_BINS):
        if low <= sim < high: return i
    return 4

In [10]:
df_train['Regime_Norm'] = df_train['Mass_Regime'].apply(get_regime)
df_train['Sim_Bin_Idx'] = df_train['cosine_similarity'].apply(assign_sim_bin)
df_mona['Regime_Norm'] = df_mona['Mass_Regime'].apply(get_regime)
df_mona['Sim_Bin_Idx'] = df_mona['cosine_similarity'].apply(assign_sim_bin)

In [12]:
df_mona.head()

,name_main,name_sub,cosine_similarity,mass_difference,Mass_Regime,label,Regime_Norm,Sim_Bin_Idx
0,MoNA_27032,MoNA_78252,0.459223,41.980000,3. Medium (10-50 Da),0,3. Medium,2
1,MoNA_66621,MoNA_92123,0.701804,46.005463,3. Medium (10-50 Da),1,3. Medium,3
2,MoNA_65585,MoNA_83619,0.505656,238.250732,5. Huge (>100 Da),0,5. Huge,2
3,MoNA_75789,MoNA_76066,0.863607,401.193734,5. Huge (>100 Da),1,5. Huge,4
4,MoNA_64797,MoNA_91816,0.104762,276.120880,5. Huge (>100 Da),0,5. Huge,0


In [14]:
# --- 3. DIVERSITY SAMPLER ---
def undersample_diverse(df_subset, target_n):
    if len(df_subset) <= target_n: return df_subset
    scaffolds = [get_scaffold_from_spec(sid) for sid in df_subset['name_main']]
    df_temp = df_subset.copy()
    df_temp['TEMP_SCAFFOLD'] = scaffolds
    groups = df_temp.groupby('TEMP_SCAFFOLD')
    group_keys = list(groups.groups.keys())
    np.random.shuffle(group_keys)
    selected_indices = []
    group_iters = [iter(groups.get_group(k).index) for k in group_keys]
    while len(selected_indices) < target_n:
        active_iters = []
        for it in group_iters:
            try:
                idx = next(it)
                selected_indices.append(idx)
                active_iters.append(it)
                if len(selected_indices) >= target_n: break
            except StopIteration: pass
        group_iters = active_iters
        if not group_iters: break
    return df_temp.loc[selected_indices].drop(columns=['TEMP_SCAFFOLD'])

In [15]:
# --- 4. EXECUTE DYNAMIC SHAPING ---
final_dfs = []
regimes = sorted(df_train['Regime_Norm'].unique())

print(f"\n--- Starting Dynamic Oval Shaping ---")

for regime in regimes:
    print(f"\n=== Regime: {regime} ===")
    
    # STEP A: Calculate the "Peak Capacity" for this Regime
    # We look at bins 2 (0.4-0.6) and 3 (0.6-0.8) in BOTH datasets to see what's physically possible.
    
    # Original counts
    orig_counts = df_train[df_train['Regime_Norm'] == regime]['Sim_Bin_Idx'].value_counts()
    # MoNA counts
    mona_counts = df_mona[df_mona['Regime_Norm'] == regime]['Sim_Bin_Idx'].value_counts()
    
    # Calculate Total Available in Hard Zones
    available_bin2 = orig_counts.get(2, 0) + mona_counts.get(2, 0)
    available_bin3 = orig_counts.get(3, 0) + mona_counts.get(3, 0)
    
    # The Peak is limited by the availability of hard data
    # We take the average availability of the hard bins, clamped by global min/max
    raw_capacity = (available_bin2 + available_bin3) / 2
    peak_target = int(min(max(raw_capacity, GLOBAL_MIN_PEAK), GLOBAL_MAX_PEAK))
    
    print(f"   > Hard Data Available: Bin2={available_bin2}, Bin3={available_bin3}")
    print(f"   > Dynamic Peak Target set to: {peak_target}")

    # STEP B: Fill Bins according to Ratios
    for bin_idx, (low, high) in enumerate(SIM_BINS):
        ratio = SHAPE_RATIOS[bin_idx]
        target_count = int(peak_target * ratio)
        
        # 1. Filter Original
        subset = df_train[(df_train['Regime_Norm'] == regime) & (df_train['Sim_Bin_Idx'] == bin_idx)]
        current_count = len(subset)
        deficit = target_count - current_count
        
        print(f"     [Sim {low}-{high:.1f}] Target: {target_count} | Found: {current_count}", end="")
        
        cell_result = None
        
        # CASE A: Too many? Undersample Diverse
        if deficit < 0:
            print(f" -> Undersampling (Diverse)")
            cell_result = undersample_diverse(subset, target_count)
            
        # CASE B: Too few? Fill from MoNA
        else:
            print(f" -> Filling from MoNA")
            cell_result = subset
            
            if deficit > 0:
                mona_candidates = df_mona[(df_mona['Regime_Norm'] == regime) & (df_mona['Sim_Bin_Idx'] == bin_idx)]
                available = len(mona_candidates)
                take = min(deficit, available)
                
                if take > 0:
                    added = mona_candidates.sample(n=take, random_state=42)
                    common_cols = list(set(df_train.columns) & set(added.columns))
                    added = added[common_cols]
                    cell_result = pd.concat([cell_result, added])
                else:
                    if available == 0: print(" (No MoNA data)", end="")

        print(f" -> Final: {len(cell_result)}")
        final_dfs.append(cell_result)


--- Starting Dynamic Oval Shaping ---

=== Regime: 0. Exact Isomers ===
   > Hard Data Available: Bin2=643, Bin3=3238
   > Dynamic Peak Target set to: 1940
     [Sim 0.0-0.2] Target: 485 | Found: 27 -> Filling from MoNA
 -> Final: 83
     [Sim 0.2-0.4] Target: 970 | Found: 138 -> Filling from MoNA
 -> Final: 202
     [Sim 0.4-0.6] Target: 1940 | Found: 436 -> Filling from MoNA
 -> Final: 643
     [Sim 0.6-0.8] Target: 1940 | Found: 2250 -> Undersampling (Diverse)
 -> Final: 1940
     [Sim 0.8-1.0] Target: 640 | Found: 7451 -> Undersampling (Diverse)
 -> Final: 640

=== Regime: 1. Isobaric ===
   > Hard Data Available: Bin2=949, Bin3=2335
   > Dynamic Peak Target set to: 1642
     [Sim 0.0-0.2] Target: 410 | Found: 147 -> Filling from MoNA
 -> Final: 352
     [Sim 0.2-0.4] Target: 821 | Found: 219 -> Filling from MoNA
 -> Final: 554
     [Sim 0.4-0.6] Target: 1642 | Found: 285 -> Filling from MoNA
 -> Final: 949
     [Sim 0.6-0.8] Target: 1642 | Found: 642 -> Filling from MoNA
 -> Fina

In [16]:
# --- 5. SAVE ---
print("\n--- Finalizing ---")
df_final = pd.concat(final_dfs, axis=0, ignore_index=True)
df_final = df_final.sample(frac=1.0, random_state=42).reset_index(drop=True)

print(f"Total Dataset Size: {len(df_final)}")
df_final.to_feather(OUTPUT_PATH)
print(f"Saved to: {OUTPUT_PATH}")


--- Finalizing ---
Total Dataset Size: 106106
Saved to: /data/nas-gpu/wang/tmach007/SpectralSimilarityPredictor/data_splits/stratified_binary_07_dataset_train_CUSTOM_BALANCED.feather
